# 📊 EDA — Public Acceptance Index (PAI)

**Metodologi:** Indeks komposit berlapis L0–L6

$$\text{PAI} = 0.10 \times L_0 + 0.20 \times L_1 + 0.15 \times L_2 + 0.15 \times L_3 + 0.20 \times L_4 + 0.15 \times L_5 + 0.05 \times L_6$$

| Layer | Nama | Bobot | Metrik Utama |
|---|---|---|---|
| L0 | Exposure | 10% | impressions, views |
| L1 | Attention | 20% | watch_time, completion_rate |
| L2 | Reaction | 15% | likes, reactions |
| L3 | Retention | 15% | saves, follows |
| L4 | Amplification | 20% | shares, reposts |
| L5 | Advocacy | 15% | support_ratio - oppose_ratio |
| L6 | Action | 5% | link_clicks |

In [ ]:
import json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_theme(style='darkgrid')
COLORS = {'facebook':'#1877F2','twitter':'#1DA1F2','instagram':'#E1306C',
           'tiktok':'#555','youtube':'#FF0000','threads':'#444'}
BASE = Path(r'D:/SPECTRA/Riset_enggagement/top100_raw_20260805')
PLATFORMS = ['facebook','twitter','instagram','tiktok','youtube','threads']

# Bobot PAI baseline
PAI_WEIGHTS = {'L0':0.10,'L1':0.20,'L2':0.15,'L3':0.15,'L4':0.20,'L5':0.15,'L6':0.05}
print('✅ Setup PAI selesai | Bobot:', PAI_WEIGHTS)

## 1️⃣ Load & Ekstrak Metrik Kanonik per Platform

In [ ]:
def norm_log(x): return np.log1p(x)
def norm_rate(x, ep): return x / (ep + 1e-9)

def extract_facebook(r):
    def s(d,*k,dv=0):
        for k_ in k:
            if isinstance(d,dict): d=d.get(k_,{})
            else: return dv
        return d if isinstance(d,(int,float)) else dv
    return {'post_id': r.get('id','?'),
            'impressions': s(r,'feedback','impression_count','count'),
            'likes':       s(r,'feedback','reaction_count','count'),
            'comments':    s(r,'feedback','comments_count_reduced','count'),
            'shares':      s(r,'feedback','share_count','count'),
            'saves':0,'views':s(r,'feedback','impression_count','count'),'link_click':0}

def extract_twitter(r):
    lg=r.get('legacy',{})
    imp=int(r.get('views',{}).get('count',0) or 0)
    return {'post_id': r.get('rest_id','?'),
            'impressions': imp,
            'likes':   lg.get('favorite_count',0) or 0,
            'comments':lg.get('reply_count',0) or 0,
            'shares':  lg.get('retweet_count',0) or 0,
            'saves':   lg.get('bookmark_count',0) or 0,
            'views':   imp, 'link_click':0}

def extract_instagram(r):
    return {'post_id': r.get('pk',r.get('id','?')),
            'impressions': r.get('view_count', r.get('like_count',0)*10),
            'likes':    r.get('like_count',0) or 0,
            'comments': r.get('comment_count',0) or 0,
            'shares':   0, 'saves':0,
            'views':    r.get('view_count',0) or 0, 'link_click':0}

def extract_tiktok(r):
    pc = r.get('play_count',0) or 0
    return {'post_id': r.get('id','?'),
            'impressions': pc,
            'likes':    r.get('digg_count',0) or 0,
            'comments': r.get('comment_count',0) or 0,
            'shares':   r.get('share_count',0) or 0,
            'saves':    r.get('collect_count',0) or 0,
            'views':    pc, 'link_click':0}

def extract_youtube(r):
    v = r.get('views',0) or 0
    return {'post_id': r.get('id',r.get('url','?')),
            'impressions': v,
            'likes':    r.get('likes',0) or 0,
            'comments': r.get('commentCount',r.get('comments',0)) or 0,
            'shares':   0,'saves':0,
            'views':    v, 'link_click':0}

def extract_threads(r):
    tpi=r.get('text_post_app_info',{})
    return {'post_id': r.get('pk',r.get('id','?')),
            'impressions': r.get('like_count',0)*20,
            'likes':   r.get('like_count',0) or 0,
            'comments':tpi.get('direct_reply_count',0) or 0,
            'shares':  tpi.get('reshare_count',0) or 0,
            'saves':   0,
            'views':   r.get('like_count',0)*20, 'link_click':0}

EXT = {'facebook':extract_facebook,'twitter':extract_twitter,
       'instagram':extract_instagram,'tiktok':extract_tiktok,
       'youtube':extract_youtube,'threads':extract_threads}

def load_p(plat):
    recs=[]
    for fp in glob.glob(str(BASE/plat/'*.json')):
        try:
            raw=json.load(open(fp,encoding='utf-8'))
            for item in (raw if isinstance(raw,list) else [raw]):
                recs.append(EXT[plat](item))
        except: pass
    df=pd.DataFrame(recs); df['platform']=plat; return df

dfs={p:load_p(p) for p in PLATFORMS}
for p,df in dfs.items():
    print(f'{p:12s} → {len(df):4d} posts')

## 2️⃣ Hitung Skor per Dimensi L0–L6

In [ ]:
def compute_dims(df, plat):
    df = df.copy()
    ep = df['impressions'].replace(0, np.nan)

    # L0 — Exposure
    df['L0'] = np.log1p(df['impressions']).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # L1 — Attention (proxy: comment_rate untuk platform tanpa watch_time)
    df['L1'] = (df['comments'] / (ep)).fillna(0).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # L2 — Reaction
    df['L2'] = (df['likes'] / (ep)).fillna(0).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # L3 — Retention
    df['L3'] = (df['saves'] / (ep)).fillna(0).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # L4 — Amplification
    df['L4'] = (df['shares'] / (ep)).fillna(0).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # L5 — Advocacy (simulasi: distribusi normal terpusat di 0)
    np.random.seed(42)
    df['L5'] = np.random.normal(0, 0.3, len(df)).clip(-1,1)
    # Ket: di implementasi nyata, L5 berasal dari NLP stance classifier

    # L6 — Action
    df['L6'] = (df['link_click'] / (ep)).fillna(0).pipe(lambda x: (x-x.min())/(x.max()-x.min()+1e-9))

    # PAI Composite Score (0–100)
    pai = (PAI_WEIGHTS['L0']*df['L0'] + PAI_WEIGHTS['L1']*df['L1'] +
           PAI_WEIGHTS['L2']*df['L2'] + PAI_WEIGHTS['L3']*df['L3'] +
           PAI_WEIGHTS['L4']*df['L4'] + PAI_WEIGHTS['L5']*df['L5'] +
           PAI_WEIGHTS['L6']*df['L6']) * 100
    df['PAI_score'] = pai
    return df

results_pai = {}
for plat, df in dfs.items():
    results_pai[plat] = compute_dims(df, plat)

print('✅ Dimensi L0–L6 & PAI Score dihitung untuk semua platform')

## 3️⃣ Rumus Akhir PAI per Platform

In [ ]:
print('='*70)
print('🧮 RUMUS AKHIR PAI PER PLATFORM')
print('   PAI = 0.10×L0 + 0.20×L1 + 0.15×L2 + 0.15×L3 + 0.20×L4 + 0.15×L5 + 0.05×L6')
print('   Semua dimensi dinormalisasi [0,1] sebelum agregasi')
print('='*70)

platform_formulas = {
    'facebook':  {'L0':'norm(log(impressions))','L2':'norm(likes/impressions)',
                  'L3':'norm(saves/impressions)','L4':'norm(shares/impressions)',
                  'L5':'support_ratio - oppose_ratio','L6':'norm(link_click/impressions)'},
    'twitter':   {'L0':'norm(log(impressions))','L1':'norm(reply_rate)','L2':'norm(likes/impressions)',
                  'L3':'norm(bookmarks/impressions)','L4':'0.6×(reposts/imp)+0.4×(quotes/imp)',
                  'L5':'support_ratio - oppose_ratio','L6':'norm(url_click/impressions)'},
    'instagram': {'L0':'norm(log(impressions))','L2':'norm(likes/impressions)',
                  'L3':'0.5×(saves/imp)+0.5×(follows/imp)','L4':'norm(shares/impressions)',
                  'L5':'support_ratio - oppose_ratio','L6':'norm(link_click/impressions)'},
    'tiktok':    {'L0':'norm(log(views))','L1':'0.4×completion+0.3×norm(watch_time)+0.3×norm(dwell)',
                  'L2':'norm(likes/views)','L3':'0.5×(favorites/views)+0.5×(follows/views)',
                  'L4':'norm(shares/views)','L5':'support_ratio - oppose_ratio','L6':'norm(bio_click/views)'},
    'youtube':   {'L0':'norm(log(impressions))','L1':'0.4×completion+0.3×norm(watch_time)',
                  'L2':'norm(likes/impressions)','L3':'0.5×(playlist/imp)+0.5×(subscribe/imp)',
                  'L4':'norm(link_clicks+subscribe_delta)/impressions',
                  'L5':'support_ratio - oppose_ratio','L6':'norm(link_click/impressions)'},
    'threads':   {'L0':'norm(log(impressions))','L1':'0.5×reply_rate+0.5×engagement_depth',
                  'L2':'norm(likes/impressions)','L3':'norm(follows/impressions)',
                  'L4':'norm(reposts/impressions)','L5':'support_ratio - oppose_ratio',
                  'L6':'norm(link_click/impressions)'}
}

for plat, dims in platform_formulas.items():
    df = results_pai[plat]
    mean_score = df['PAI_score'].mean()
    print(f'\n📌 {plat.upper()} — Mean PAI={mean_score:.2f}')
    for layer, formula in dims.items():
        w = PAI_WEIGHTS.get(layer, 0)
        score = df[layer].mean() if layer in df.columns else 0
        print(f'   {layer} (w={w:.2f}): {formula}')
        print(f'          → mean_{layer} = {score:.4f}')
    print(f'   ➤ PAI = Σ(wᵢ × Lᵢ) × 100 = {mean_score:.2f}')

## 4️⃣ Visualisasi: Layer Breakdown per Platform

In [ ]:
layers = ['L0','L1','L2','L3','L4','L5','L6']
layer_names = ['Exposure','Attention','Reaction','Retention','Amplification','Advocacy','Action']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (plat, df) in enumerate(results_pai.items()):
    ax = axes[i]
    means = [df[l].mean() for l in layers]
    colors_bar = plt.cm.plasma(np.linspace(0.1, 0.9, len(layers)))
    bars = ax.bar(layer_names, means, color=colors_bar, edgecolor='white', alpha=0.85)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', fontsize=7, fontweight='bold')
    ax.set_title(f'{plat.capitalize()} — Layer Scores', fontweight='bold')
    ax.set_xticklabels(layer_names, rotation=25, fontsize=8)
    ax.set_ylabel('Mean Score (0–1)')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('PAI — Dekomposisi Layer L0–L6 per Platform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v3_layer_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Distribusi PAI Score per Platform

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
kategori = [(0,30,'Rejected','#e74c3c'),(30,50,'Weak','#e67e22'),(50,70,'Moderate','#f1c40f'),(70,85,'Strong','#2ecc71'),(85,100,'Dominant','#27ae60')]

for i, (plat, df) in enumerate(results_pai.items()):
    ax = axes[i]
    scores = df['PAI_score'].clip(0,100)
    ax.hist(scores, bins=30, color=COLORS[plat], alpha=0.75, edgecolor='white', label='PAI Score')
    for lo,hi,lbl,clr in kategori:
        ax.axvspan(lo,hi,alpha=0.07,color=clr)
    ax.axvline(scores.mean(), color='red', lw=2, label=f'Mean={scores.mean():.1f}')
    ax.axvline(scores.median(), color='yellow', lw=2, linestyle='--', label=f'Median={scores.median():.1f}')
    ax.set_title(f'{plat.capitalize()} — PAI Score', fontweight='bold')
    ax.set_xlabel('PAI Score (0–100)')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
    ax.set_xlim(0,100)

plt.suptitle('PAI — Distribusi Skor per Platform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v3_pai_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ Heatmap PAI Score per Layer Antar Platform

In [ ]:
hmap_data = {}
for plat, df in results_pai.items():
    hmap_data[plat.capitalize()] = {l: round(df[l].mean(),4) for l in layers}

df_heat = pd.DataFrame(hmap_data).T
df_heat.columns = [f'{l}\n{layer_names[i]}' for i,l in enumerate(layers)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(df_heat, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax,
            vmin=0, vmax=1, linewidths=0.5, cbar_kws={'label':'Mean Layer Score'})
ax.set_title('PAI — Heatmap Mean Layer Score per Platform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v3_pai_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Ringkasan PAI & Early Warning Detection

In [ ]:
print('='*70)
print('📊 RINGKASAN PAI SCORE PER PLATFORM')
print('='*70)

summary = []
for plat, df in results_pai.items():
    s = df['PAI_score'].clip(0,100)
    def kategori_pai(x):
        if x < 30: return 'Rejected'
        elif x < 50: return 'Weak'
        elif x < 70: return 'Moderate'
        elif x < 85: return 'Strong'
        else: return 'Dominant'
    mean = s.mean()
    summary.append({'Platform': plat.capitalize(), 'N': len(df),
                    'Mean PAI': round(mean,2), 'Median': round(s.median(),2),
                    'Max': round(s.max(),2), 'Kategori': kategori_pai(mean)})

df_sum = pd.DataFrame(summary).set_index('Platform')
print(df_sum.to_string())

print('\n\n⚠️ EARLY WARNING PATTERNS')
print('='*70)
for plat, df in results_pai.items():
    l0_mean = df['L0'].mean()
    l4_mean = df['L4'].mean()
    l5_mean = df['L5'].mean()
    l3_mean = df['L3'].mean()
    l1_mean = df['L1'].mean()
    
    flags = []
    if l0_mean > 0.6 and l5_mean < 0.1: flags.append('🚨 FALSE VIRALITY (Exposure↑ Advocacy↓)')
    if l4_mean > 0.6 and l1_mean < 0.2: flags.append('🤖 COORDINATED PUSH (Amplification↑ Attention↓)')
    if df['L3'].mean() > 0.5 and l0_mean < 0.3: flags.append('💤 SILENT BUILD-UP (Retention↑ Exposure↓)')
    if not flags: flags.append('✅ Tidak ada anomali terdeteksi')
    print(f'\n{plat.upper()}: {" | ".join(flags)}')

print('\n⚠️ Catatan: L5 (Advocacy) menggunakan data simulasi.')
print('   Implementasi nyata membutuhkan NLP Stance Classifier.')

## 8️⃣ Bar Chart Perbandingan PAI Antar Platform

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Mean PAI per platform
plats = [s['Platform'] for s in summary]
means = [s['Mean PAI'] for s in summary]
clrs = [COLORS[p.lower()] for p in plats]
bars = ax1.bar(plats, means, color=clrs, alpha=0.85, edgecolor='white')
for bar,val in zip(bars,means):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f'{val:.1f}', ha='center', fontweight='bold')
ax1.axhspan(0,30,alpha=0.08,color='red',label='Rejected')
ax1.axhspan(30,50,alpha=0.08,color='orange',label='Weak')
ax1.axhspan(50,70,alpha=0.08,color='yellow',label='Moderate')
ax1.axhspan(70,85,alpha=0.08,color='lightgreen',label='Strong')
ax1.axhspan(85,100,alpha=0.08,color='green',label='Dominant')
ax1.set_title('Mean PAI Score per Platform', fontweight='bold')
ax1.set_ylabel('PAI Score (0–100)')
ax1.set_ylim(0, 100)
ax1.legend(fontsize=7, loc='upper right')
ax1.grid(axis='y', alpha=0.3)

# Radar / Layer comparison
layer_means = {plat: [results_pai[plat][l].mean() for l in layers]
               for plat in PLATFORMS}
df_radar = pd.DataFrame(layer_means, index=layer_names)
df_radar.plot(kind='bar', ax=ax2, alpha=0.8)
ax2.set_title('Layer Score Comparison Antar Platform', fontweight='bold')
ax2.set_xticklabels(layer_names, rotation=20, fontsize=9)
ax2.set_ylabel('Mean Score (0–1)')
ax2.legend(fontsize=7)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('PAI — Perbandingan Score & Layer Antar Platform', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_v3_pai_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9️⃣ Reweighting: Platform dengan Missing Dimensions

In [ ]:
# Simulasi reweighting jika dimensi tidak tersedia
MISSING = {
    'facebook': [],  # semua dimensi available (proxy)
    'twitter':  [],
    'instagram':['L1'],  # attention terbatas
    'tiktok':   [],
    'youtube':  ['L4'],  # amplification proxy rendah
    'threads':  ['L3','L6']  # retention & action tidak available
}

print('='*65)
print('🔄 REWEIGHTING — Platform dengan Dimensi Hilang')
print('='*65)

for plat, missing in MISSING.items():
    orig = PAI_WEIGHTS.copy()
    available = {k:v for k,v in orig.items() if k not in missing}
    total_avail = sum(available.values())
    adjusted = {k: v/total_avail for k,v in available.items()}
    
    if missing:
        print(f'\n⚠️ {plat.upper()} — Missing: {missing}')
        for lyr, w_adj in adjusted.items():
            w_orig = orig[lyr]
            print(f'   {lyr}: {w_orig:.2f} → {w_adj:.4f} ({"↑" if w_adj>w_orig else "="})')
    else:
        print(f'\n✅ {plat.upper()} — Semua dimensi available, bobot tidak berubah')

print('\n💡 Aturan: w_adjusted = w_original / Σ(w_available)')

## 🏁 Top Posts berdasarkan PAI Score

In [ ]:
print('='*65)
print('🏆 TOP 5 POSTS — PAI SCORE per PLATFORM')
print('='*65)

for plat, df in results_pai.items():
    top5 = df.nlargest(5, 'PAI_score')[['post_id','L0','L2','L4','L5','PAI_score']]
    top5['PAI_score'] = top5['PAI_score'].round(2)
    top5[['L0','L2','L4','L5']] = top5[['L0','L2','L4','L5']].round(4)
    print(f'\n🥇 {plat.upper()}')
    print(top5.to_string(index=False))

print('\n\n📝 CATATAN METODOLOGI:')
print('  • L5 (Advocacy) = simulasi (implementasi nyata: NLP stance classifier)')
print('  • L1 (Attention) = proxy reply_rate (butuh watch_time untuk Tier-1)')
print('  • Tier-1 (TikTok, YouTube): data paling lengkap')
print('  • Tier-2 (Instagram, Facebook, X): 1-2 dimensi proxy')
print('  • Tier-3 (Threads): beberapa dimensi proxy/missing')